# Objetivo de Notebook

Esta notebook implementa un pipeline robusto para modelado no supervisado, separando claramente los datos de entrenamiento de los datos de evaluación, asegurando que el modelo no vea los mismos pozos/etapas que luego va a analizar.

## 🤖 Split Robusto y Modelado No Supervisado - CCL Anomalías

Esta notebook implementa:
- División del dataset por pozo y etapa.
- Entrenamiento del modelo (Isolation Forest) en runs seleccionadas.
- Evaluación del modelo en datos no vistos (evaluación visual de anomalías).
- Preparación de los resultados para visualización.

Esta estrategia permite evitar sobreajuste y simular condiciones de operación real.


### 💾 Celda 2 – Carga del dataset enriquecido

In [1]:
import pandas as pd

# Cargar dataset con features + CCL + TENS
df = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_features.csv")
df = df.sort_values(by=["pozo", "etapa", "DEPT"]).reset_index(drop=True)

print(f"Total registros: {len(df)}")
df.head()


Total registros: 97242


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,CCL_norm_mean,CCL_norm_std,CCL_norm_max,CCL_norm_min,abs_dCCL_mean,abs_dCCL_std,abs_dCCL_max,TENS_mean,TENS_std,TENS_max
0,2800.05,0.00124,2626.00001,BPE-2343_E36_Up__21Nov24_232516.las,BPE-2343,Up,E36,0.248248,NaN,NaN,...,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993
1,2800.10,-0.00802,2626.00001,BPE-2343_E36_Up__21Nov24_232516.las,BPE-2343,Up,E36,-1.605606,-1.853854,1.853854,...,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993
2,2800.15,-0.01364,2628.26833,BPE-2343_E36_Up__21Nov24_232516.las,BPE-2343,Up,E36,-2.730731,-1.125125,1.125125,...,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993
3,2800.20,-0.01094,2644.00046,BPE-2343_E36_Up__21Nov24_232516.las,BPE-2343,Up,E36,-2.190190,0.540541,0.540541,...,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993
4,2800.25,-0.00511,2644.00046,BPE-2343_E36_Up__21Nov24_232516.las,BPE-2343,Up,E36,-1.023023,1.167167,1.167167,...,0.138756,3.094922,10.0,-9.997998,1.296902,1.527585,19.815816,2301.805072,252.059941,2816.99993


### 📋 Celda 3 – Generar resumen de etapas por pozo

In [2]:
df_summary = df[["pozo", "etapa"]].drop_duplicates().sort_values(["pozo", "etapa"])
print(f"Total de runs únicas: {len(df_summary)}")
df_summary.head(49)


Total de runs únicas: 4


,pozo,etapa
0,BPE-2343,E36
30747,BPE-2343,E38
59614,BPE-2343,E46
79311,BPE-2343,E48


### ✂️ Celda 4 – Split robusto por combinación pozo-etapa

In [8]:
# ⚙️ Definí manualmente qué pozo(s) y etapa(s) usar
train_seleccion = [
    ("BPE-2343", "E36"),
    ("BPE-2343", "E48"),
    ("BPE-2343", "E46"),
]

eval_seleccion = [
    ("BPE-2343", "E38"),
]

# Separar el dataset según las combinaciones seleccionadas
df_train = df[df[["pozo", "etapa"]].apply(tuple, axis=1).isin(train_seleccion)].copy()
df_eval = df[df[["pozo", "etapa"]].apply(tuple, axis=1).isin(eval_seleccion)].copy()

print(f"✅ Train: {df_train.shape[0]} registros en {len(train_seleccion)} runs")
print(f"✅ Eval:  {df_eval.shape[0]} registros en {len(eval_seleccion)} runs")

✅ Train: 68375 registros en 3 runs
✅ Eval:  28867 registros en 1 runs


### 🧠 Celda 5 – Entrenamiento con Isolation Forest

In [9]:
from sklearn.ensemble import IsolationForest

# Seleccionar features numéricas (excluimos DEPT y cualquier score anterior)
exclude = ["DEPT", "score_iso", "anomaly_iso"]
features_cols = [col for col in df_train.select_dtypes(include='number').columns if col not in exclude]

X_train = df_train[features_cols].fillna(0)
X_eval = df_eval[features_cols].fillna(0)

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
iso.fit(X_train)

# Aplicar a evaluación
df_eval["score_iso"] = -iso.decision_function(X_eval)
df_eval["anomaly_iso"] = iso.predict(X_eval)  # -1 = anomalía, 1 = normal


### 📦 Celda 6 – Guardar resultados de evaluación

In [10]:
df_eval.to_csv(r"C:\Developer\fundamentos\data\ccl_eval_scores.csv", index=False)
print("✅ Resultados de evaluación exportados a ccl_eval_scores.csv")


✅ Resultados de evaluación exportados a ccl_eval_scores.csv


### 📊 Celda 7 – Vista rápida de anomalías detectadas

In [11]:
anomalias_por_etapa = df_eval[df_eval["anomaly_iso"] == -1].groupby(["pozo", "etapa"]).size().reset_index(name="anomalías")
anomalias_por_etapa.sort_values("anomalías", ascending=False).head(10)


,pozo,etapa,anomalías
0,BPE-2343,E38,417
